# End-to-End FINN Flow for a Simple Convolutional Net
-----------------------------------------------------------------

In this notebook, we will go through the FINN steps needed to take a binarized convolutional network all the way down to a heterogeneous streaming dataflow accelerator running on the FPGA. 

It's recommended to go through the simpler [end-to-end notebook for a fully connected network](tfc_end2end_example.ipynb) first, since many steps here are very similar and we will focus on what is done differently for convolutions.

This notebook is quite lengthy, and some of the cells (involving Vivado synthesis) may take up to an hour to finish running. To let you save and resume your progress, we will save the intermediate ONNX models that are generated in the various steps to disk, so that you can jump back directly to where you left off.

In [1]:
import os, shutil

print("FINN_BUILD_DIR =", os.environ.get("FINN_BUILD_DIR"))
print("HLS_PATH =", os.environ.get("HLS_PATH"))
print("VIVADO_PATH =", os.environ.get("VIVADO_PATH"))
print("vitis_hls =", shutil.which("vitis_hls"))
print("vivado =", shutil.which("vivado"))

assert shutil.which("vitis_hls") is not None
assert shutil.which("vivado") is not None

FINN_BUILD_DIR = /tmp/finn_dev_kinanah
HLS_PATH = /tools/Xilinx/Vitis_HLS/2022.2
VIVADO_PATH = /tools/Xilinx/Vivado/2022.2
vitis_hls = /tools/Xilinx/Vitis_HLS/2022.2/bin/vitis_hls
vivado = /tools/Xilinx/Vivado/2022.2/bin/vivado


## Quick Introduction to the CNV-w1a1 Network

The particular quantized neural network (QNN) we will be targeting in this notebook is referred to as CNV-w1a1 and it classifies 32x32 RGB images into one of ten CIFAR-10 classes. All weights and activations in this network are quantized to bipolar values (either -1 or +1), with the exception of the input (which is RGB with 8 bits per channel) and the final output (which is 32-bit numbers). It first appeared in the original [FINN paper](https://arxiv.org/abs/1612.07119) from ISFPGA'17 with the name CNV, as a variant of the binarized convolutional network from the [BinaryNet paper](https://arxiv.org/abs/1602.02830), in turn inspired by the VGG-11 topology which was the runner-up for the 2014 [ImageNet Large Scale Visual Recognition Challenge](http://www.image-net.org/challenges/LSVRC/).


You'll have a chance to interactively examine the layers that make up the network in Netron in a moment, so that's enough about the network for now. 

## Quick Recap of the End-to-End Flow

The FINN compiler comes with many *transformations* that modify the ONNX representation of the network according to certain patterns. This notebook will demonstrate a *possible* sequence of such transformations to take a particular trained network all the way down to hardware, as shown in the figure below.

![](finn-design-flow-example.svg)

The white fields show the state of the network representation in the respective step. The colored fields represent the transformations that are applied to the network to achieve a certain result. The diagram is divided into 5 sections represented by a different color, each of it includes several flow steps. The flow starts in top left corner with Brevitas export (green section), followed by the preparation of the network (blue section) to bring the network into a form in which each layer can be represented by either a Vitis HLS function or a Verilog module. The model then gets passed to Vivado IPI stitching (orange section), and finally a PYNQ overlay bitfile is built and can be tested on a PYNQ board (yellow section).
There is an additional section for functional verification (red section) on the right side of the diagram, which we will not cover in this notebook. For details please take a look in the verification notebook which you can find [here](tfc_end2end_verification.ipynb)


We will use the helper function `showInNetron` to show the ONNX model at the current transformation step. The Netron displays are interactive, but they only work when running the notebook actively and not on GitHub (i.e. if you are viewing this on GitHub you'll only see blank squares).

In [2]:
import os
print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])

FINN_BUILD_DIR = /tmp/finn_dev_kinanah


In [3]:
from finn.util.visualization import showInNetron
import os

base_build_dir = os.environ["FINN_BUILD_DIR"]

build_dir = os.path.join(base_build_dir, "cnv_folding_C")

os.makedirs(build_dir, exist_ok=True)

print("Build dir =", build_dir)
print("Base FINN build dir =", base_build_dir)

Build dir = /tmp/finn_dev_kinanah/cnv_folding_C
Base FINN build dir = /tmp/finn_dev_kinanah


## 1. Brevitas Export, FINN Import and Tidy-Up

Similar to what we did in the TFC-w1a1 end-to-end notebook, we will start by exporting the [pretrained CNV-w1a1 network](https://github.com/Xilinx/brevitas/tree/master/src/brevitas_examples/bnn_pynq) to ONNX, importing that into FINN and running the "tidy-up" transformations to have a first look at the topology. The network will be exported in QONNX format and then converted into the FINN-ONNX format to prepare it for the FINN compiler.

In [4]:
import torch
import onnx
from finn.util.test import get_test_model_trained
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

cnv = get_test_model_trained("CNV", 1, 1)
export_onnx_path = build_dir + "/end2end_cnv_w1a1_export.onnx"
export_qonnx(cnv, torch.randn(1, 3, 32, 32), export_onnx_path)
qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)
model = ModelWrapper(export_onnx_path)
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())
model.save(build_dir + "/end2end_cnv_w1a1_tidy.onnx")

Now that the model is exported, let's have a look at its layer structure with Netron. Remember that the visualization below is interactive, you can click on the individual nodes and view the layer attributes, trained weights and so on.

In [5]:
showInNetron(build_dir+"/end2end_cnv_w1a1_tidy.onnx")

Serving '/tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_tidy.onnx' at http://0.0.0.0:8081


You can see that the network is composed of a repeating convolution-convolution-maxpool layer pattern to extract features using 3x3 convolution kernels (with weights binarized), followed by fully connected layers acting as the classifier. Also notice the initial `MultiThreshold` layer at the beginning of the network, which is quantizing float inputs to 8-bit ones.

### Adding Pre- and Postprocessing <a id='prepost'></a>

Preprocessing and postprocessing steps can be added directly in the ONNX graph. In this case, the preprocessing step divides the input `uint8` data by 255 so the inputs to the CNV-w1a1 network are bounded between [0, 1]. The postprocessing step takes the output of the network and returns the index (0-9) of the image category with the highest probability (top-1). 

In [6]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType

model = ModelWrapper(build_dir+"/end2end_cnv_w1a1_tidy.onnx")
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)
# preprocessing: torchvision's ToTensor divides uint8 inputs by 255
totensor_pyt = ToTensor()
chkpt_preproc_name = build_dir+"/end2end_cnv_w1a1_preproc.onnx"
export_qonnx(totensor_pyt, torch.randn(ishape), chkpt_preproc_name)
qonnx_cleanup(chkpt_preproc_name, out_file=chkpt_preproc_name)
pre_model = ModelWrapper(chkpt_preproc_name)
pre_model = pre_model.transform(ConvertQONNXtoFINN())

# join preprocessing and core model
model = model.transform(MergeONNXModels(pre_model))
# add input quantization annotation: UINT8 for all BNN-PYNQ models
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

/home/kinanah/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [7]:
from qonnx.transformation.insert_topk import InsertTopK
from qonnx.transformation.infer_datatypes import InferDataTypes

# postprocessing: insert Top-1 node at the end
model = model.transform(InsertTopK(k=1))
chkpt_name = build_dir+"/end2end_cnv_w1a1_pre_post.onnx"
# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model.save(chkpt_name)

In [9]:
showInNetron(build_dir+"/end2end_cnv_w1a1_pre_post.onnx")

Serving '/tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_pre_post.onnx' at http://0.0.0.0:8081


## 2. How FINN Implements Convolutions: Lowering and Streamlining

In FINN, we implement convolutions with the *lowering* approach: we convert them to matrix-matrix multiply operations, where one of the matrices is generated by sliding a window over the input image. You can read more about the sliding window operator and how convolution lowering works [in this notebook](https://github.com/maltanar/qnn-inference-examples/blob/master/3-convolutional-binarized-gtsrb.ipynb). The streaming dataflow architecture we will end up with is going to look something like this figure from the [FINN-R paper](https://arxiv.org/abs/1809.04570):

![](cnv-mp-fc.png)

Note how the convolution layer looks very similar to the fully connected one in terms of the matrix-vector-threshold unit (MVTU) or sometimes called matrix-vector-activation unit (MVAU). But now the MVTU is preceded by a sliding window unit that produces the matrix from the input image. All of these building blocks, including the `MaxPool` layer you see in this figure, exist as templated Vitis HLS C++ functions in [finn-hlslib](https://github.com/Xilinx/finn-hlslib) and/or as RTL modules in [finn-rtllib](https://github.com/Xilinx/finn/tree/main/finn-rtllib).


To target this kind of hardware architecture with our network we'll apply a convolution lowering transformation, in addition to streamlining. You may recall the *streamlining transformation* that we applied to the TFC-w1a1 network, which is a series of mathematical simplifications that allow us to get rid of floating point scaling operations by implementing few-bit activations as thresholding operations. 

**The current implementation of streamlining is highly network-specific and may not work for your network if its topology is very different than the example network here. We hope to rectify this in future releases.**

In [10]:
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

model = ModelWrapper(build_dir + "/end2end_cnv_w1a1_pre_post.onnx")
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(Streamline())
# absorb final add-mul nodes into TopK
model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model.save(build_dir + "/end2end_cnv_w1a1_streamlined.onnx")

We won't go into too much detail about what happens in each transformation and why they are called in the particular order they are (feel free to visualize the intermediate steps using Netron yourself if you are curious) but here is a brief summary:

* `Streamline` moves floating point scaling and addition operations closer to the input of the nearest thresholding activation and absorbs them into thresholds
* `LowerConvsToMatMul` converts ONNX `Conv` nodes into sequences of `Im2Col, MatMul` nodes as discussed above. `Im2Col` is a custom FINN ONNX high-level node type that implements the sliding window operator.
* `MakeMaxPoolNHWC` and `AbsorbTransposeIntoMultiThreshold` convert the *data layout* of the network into the NHWC data layout that finn-hlslib and finn-rtllib primitives use. NCHW means the tensor dimensions are ordered as `(N : batch, H : height, W : width, C : channels)` (assuming 2D images). The ONNX standard ops normally use the NCHW layout, but the ONNX intermediate representation itself does not dictate any data layout.
* You may recall `ConvertBipolarMatMulToXnorPopcount` from the TFC-w1a1 example, which is needed to implement bipolar-by-bipolar (w1a1) networks correctly using finn-hlslib.

Let's visualize the streamlined and lowered network with Netron. Observe how all the `Conv` nodes have turned into pairs of `Im2Col, MatMul` nodes, and many nodes including `BatchNorm, Mul, Add` nodes have disappeared and replaced with `MultiThreshold` nodes.

In [12]:
showInNetron(build_dir+"/end2end_cnv_w1a1_streamlined.onnx")

Serving '/tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_streamlined.onnx' at http://0.0.0.0:8081


## 3. Partitioning, Conversion to HW Layers and Folding

The next steps will be (again) very similar to what we did for the TFC-w1a1 network. We'll first convert the layers that we can put into the FPGA into their HW equivalents, separate them out into a *dataflow partition* and specialize them to HLS variants:


In [13]:
from finn.util.basic import pynq_part_map
# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z1"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10

import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts

model = ModelWrapper(build_dir + "/end2end_cnv_w1a1_streamlined.onnx")
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
# TopK to LabelSelect
model = model.transform(to_hw.InferLabelSelectLayer())
# input quantization (if any) to standalone thresholding
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(to_hw.InferStreamingMaxPool())
# get rid of Reshape(-1, 1) operation between hw nodes
model = model.transform(RemoveCNVtoFCFlatten())
# get rid of Tranpose -> Tranpose identity seq
model = model.transform(absorb.AbsorbConsecutiveTransposes())
# infer tensor data layouts
model = model.transform(InferDataLayouts())
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(build_dir + "/end2end_cnv_w1a1_dataflow_parent.onnx")
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = ModelWrapper(dataflow_model_filename)
dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))
dataflow_model.save(build_dir + "/end2end_cnv_w1a1_dataflow_model.onnx")

Notice the additional `RemoveCNVtoFCFlatten` transformation that was not used for TFC-w1a1. In the last Netron visualization you may have noticed a `Reshape` operation towards the end of the network where the convolutional part of the network ends and the fully-connected layers started. That `Reshape` is essentialy a tensor flattening operation, which we can remove for the purposes of hardware implementation. We can examine the contents of the dataflow partition with Netron, and observe the `ConvolutionInputGenerator`, `MatrixVectorActivation` and `StreamingMaxPool_Batch` nodes that implement the sliding window, matrix multiply and maxpool operations. *Note that the MatrixVectorActivation instances following the ConvolutionInputGenerator nodes are really implementing the convolutions, despite the name. The final three MatrixVectorActivation instances implement actual FC layers.*

In [15]:
showInNetron(build_dir + "/end2end_cnv_w1a1_dataflow_parent.onnx")

Serving '/tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_dataflow_parent.onnx' at http://0.0.0.0:8081


Note that pretty much everything has gone into the `StreamingDataflowPartition` node; the only operation remaining is to apply a `Transpose` to obtain NHWC input from a NCHW input (the ONNX default). 

In [17]:
showInNetron(build_dir + "/end2end_cnv_w1a1_dataflow_model.onnx")

Serving '/tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_dataflow_model.onnx' at http://0.0.0.0:8081


Now we have to set the *folding factors* for certain layers to adjust the performance of our accelerator, similar to the TFC-w1a1 example. We'll also set the desired FIFO depths around those layers, which are important to achieve full throughput in the accelerator.

In [18]:
import os
import numpy as np
import shutil

from qonnx.core.modelwrapper import ModelWrapper
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

from finn.util.basic import pynq_part_map

pynq_board = "Pynq-Z1"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10

c_build_dir = "/tmp/finn_dev_kinanah/cnv_folding_C"

folded_model_file = os.path.join(
    c_build_dir,
    "end2end_cnv_w1a1_folded.onnx"
)

parent_model_file = os.path.join(
    c_build_dir,
    "end2end_cnv_w1a1_dataflow_parent.onnx"
)

print("Folded model:", folded_model_file)
print("Exists:", os.path.isfile(folded_model_file))

print("Parent model:", parent_model_file)
print("Exists:", os.path.isfile(parent_model_file))

print("FPGA part:", fpga_part)

Folded model: /tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_folded.onnx
Exists: True
Parent model: /tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_dataflow_parent.onnx
Exists: True
FPGA part: xc7z020clg400-1


In [20]:
import torch

from finn.util.test import (
    get_trained_network_and_ishape,
    get_example_input,
    get_topk,
)

from finn.util.pytorch import ToTensor


def get_golden_io_pair_local(
    topology,
    wbits,
    abits,
    return_topk=None
):
    # Load the trained reference network
    model, ishape = get_trained_network_and_ishape(
        topology,
        wbits,
        abits
    )

    # Get one example input
    input_tensor_npy = get_example_input(topology)

    # Convert to torch
    input_tensor_torch = torch.from_numpy(
        input_tensor_npy
    ).float()

    # Apply the same preprocessing used by FINN
    preproc = ToTensor()
    input_tensor_torch = preproc.forward(
        input_tensor_torch
    ).detach()

    # Golden/reference output
    output_tensor_npy = (
        model.forward(input_tensor_torch)
        .detach()
        .numpy()
    )

    # CNV final output uses TopK
    if return_topk is not None:
        output_tensor_npy = get_topk(
            output_tensor_npy,
            k=return_topk
        )

    return input_tensor_npy, output_tensor_npy

In [30]:
x_verify, y_expected = get_golden_io_pair_local("cnv", 1, 1, return_topk=1)

print("Input shape:", x_verify.shape)
print("Expected output shape:", y_expected.shape)
print("Input dtype:", x_verify.dtype)
print("Expected dtype:", y_expected.dtype)
print("Expected output:", y_expected)

Input shape: (1, 3, 32, 32)
Expected output shape: (1,)
Input dtype: float32
Expected dtype: int64
Expected output: [3]


In [31]:
import os
import numpy as np
import shutil

c_build_dir = "/tmp/finn_dev_kinanah/cnv_folding_C"

verification_data_dir = os.path.join(
    c_build_dir,
    "functional_verification"
)

os.makedirs(verification_data_dir, exist_ok=True)

verify_input_file = os.path.join(
    verification_data_dir,
    "input.npy"
)

verify_expected_file = os.path.join(
    verification_data_dir,
    "expected_output.npy"
)

np.save(verify_input_file, x_verify)
np.save(verify_expected_file, y_expected)

print("Input saved:", verify_input_file)
print("Expected saved:", verify_expected_file)

print("Input exists:", os.path.isfile(verify_input_file))
print("Expected exists:", os.path.isfile(verify_expected_file))

print("Saved input shape:", np.load(verify_input_file).shape)
print("Saved expected shape:", np.load(verify_expected_file).shape)
print("Saved expected output:", np.load(verify_expected_file))

Input saved: /tmp/finn_dev_kinanah/cnv_folding_C/functional_verification/input.npy
Expected saved: /tmp/finn_dev_kinanah/cnv_folding_C/functional_verification/expected_output.npy
Input exists: True
Expected exists: True
Saved input shape: (1, 3, 32, 32)
Saved expected shape: (1,)
Saved expected output: [3]


In [32]:
verify_output_dir = os.path.join(c_build_dir,"verification_build")

intermediate_dir = os.path.join(verify_output_dir, "intermediate_models")

os.makedirs(intermediate_dir, exist_ok=True)

parent_model_file = os.path.join(c_build_dir,
    "end2end_cnv_w1a1_dataflow_parent.onnx")

verification_parent = os.path.join(intermediate_dir, "dataflow_parent.onnx")

shutil.copy2(parent_model_file, verification_parent)

print("Original parent exists:", os.path.isfile(parent_model_file))

print("Verification parent exists:", os.path.isfile(verification_parent))

print("Verification parent:", verification_parent)

Original parent exists: True
Verification parent exists: True
Verification parent: /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/intermediate_models/dataflow_parent.onnx


In [33]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

steps_verify = [
    "step_hw_codegen",
    "step_hw_ipgen",
    "step_set_fifo_depths",
    "step_create_stitched_ip",
]

cfg_verify = build_cfg.DataflowBuildConfig(
    output_dir=verify_output_dir,

    synth_clk_period_ns=target_clk_ns,
    fpga_part=fpga_part,

    # Keep the same FIFO strategy used for Configuration C
    auto_fifo_depths=False,

    # Recommended for RTL simulation
    rtlsim_use_vivado_comps=False,

    steps=steps_verify,

    # We only need the stitched design for verification
    generate_outputs=[build_cfg.DataflowOutputType.STITCHED_IP],

    # Functional verification point
    verify_steps=[build_cfg.VerificationStepType.STITCHED_IP_RTLSIM],

    verify_input_npy=verify_input_file,
    verify_expected_output_npy=verify_expected_file,

    # Required because stitched-IP verification uses dataflow_parent.onnx
    save_intermediate_models=True,

    # Don't enter pdb if something fails
    enable_build_pdb_debug=False,

    # Keep notebook output manageable
    verbose=False,)

print("Verification configuration created successfully.")

Verification configuration created successfully.


In [1]:
import time
import torch
import numpy as np

from finn.util.test import get_test_model_trained

# --------------------------------------------------
# CPU setup
# --------------------------------------------------

torch.set_num_threads(1)

print("PyTorch version:", torch.__version__)
print("CPU threads used:", torch.get_num_threads())

# Load the exact same CNV-w1a1 reference model
print("\nLoading Brevitas CNV-w1a1...")
cnv_cpu = get_test_model_trained("CNV", 1, 1)
cnv_cpu.eval()

# Batch size = 1, same input dimensions as the FPGA model
dummy_input = torch.randn(
    1, 3, 32, 32,
    dtype=torch.float32
)

# --------------------------------------------------
# Warm-up
# --------------------------------------------------

warmup_iterations = 20

with torch.no_grad():
    for _ in range(warmup_iterations):
        _ = cnv_cpu(dummy_input)

print("Warm-up completed.")

# --------------------------------------------------
# Benchmark
# --------------------------------------------------

iterations = 500
times_ms = []

with torch.no_grad():
    for _ in range(iterations):

        t0 = time.perf_counter()

        _ = cnv_cpu(dummy_input)

        t1 = time.perf_counter()

        times_ms.append(
            (t1 - t0) * 1000.0
        )

times_ms = np.array(times_ms)

# --------------------------------------------------
# Results
# --------------------------------------------------

mean_latency_ms = np.mean(times_ms)
median_latency_ms = np.median(times_ms)
min_latency_ms = np.min(times_ms)
std_latency_ms = np.std(times_ms)

cpu_fps = 1000.0 / mean_latency_ms

total_time_s = np.sum(times_ms) / 1000.0

print("\n======================================")
print("        CPU BASELINE RESULTS")
print("======================================")

print(f"Iterations              : {iterations}")
print(f"Total inference time    : {total_time_s:.4f} s")
print(f"Mean latency/image      : {mean_latency_ms:.4f} ms")
print(f"Median latency/image    : {median_latency_ms:.4f} ms")
print(f"Minimum latency         : {min_latency_ms:.4f} ms")
print(f"Latency std             : {std_latency_ms:.4f} ms")
print(f"CPU throughput          : {cpu_fps:.2f} FPS")

PyTorch version: 1.13.1+cu116
CPU threads used: 1

Loading Brevitas CNV-w1a1...


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)


Warm-up completed.

        CPU BASELINE RESULTS
Iterations              : 500
Total inference time    : 10.8031 s
Mean latency/image      : 21.6061 ms
Median latency/image    : 19.3709 ms
Minimum latency         : 14.2583 ms
Latency std             : 6.3745 ms
CPU throughput          : 46.28 FPS


In [2]:
import time
import torch
import numpy as np

from finn.util.test import get_test_model_trained

torch.set_num_threads(1)

print("Loading CNV-w1a1...")
cnv_cpu = get_test_model_trained("CNV", 1, 1)
cnv_cpu.eval()

dummy_input = torch.randn(1, 3, 32, 32)

# Warm-up
with torch.no_grad():
    for _ in range(30):
        _ = cnv_cpu(dummy_input)

runs = 5
iterations = 500

run_mean_latency = []
run_throughput = []

print("\n===== CPU BENCHMARK: 5 RUNS =====")

for run in range(runs):

    times_ms = []

    with torch.no_grad():
        for _ in range(iterations):

            t0 = time.perf_counter()
            _ = cnv_cpu(dummy_input)
            t1 = time.perf_counter()

            times_ms.append((t1 - t0) * 1000.0)

    times_ms = np.array(times_ms)

    mean_latency = np.mean(times_ms)
    fps = 1000.0 / mean_latency

    run_mean_latency.append(mean_latency)
    run_throughput.append(fps)

    print(
        f"Run {run+1}: "
        f"Latency = {mean_latency:.4f} ms, "
        f"Throughput = {fps:.2f} FPS"
    )

run_mean_latency = np.array(run_mean_latency)
run_throughput = np.array(run_throughput)

print("\n======================================")
print("        FINAL CPU RESULTS")
print("======================================")

print(f"Runs                     : {runs}")
print(f"Iterations per run       : {iterations}")

print(
    f"Average latency          : "
    f"{np.mean(run_mean_latency):.4f} ms"
)

print(
    f"Latency std between runs : "
    f"{np.std(run_mean_latency):.4f} ms"
)

print(
    f"Average throughput       : "
    f"{np.mean(run_throughput):.2f} FPS"
)

print(
    f"Throughput std           : "
    f"{np.std(run_throughput):.2f} FPS"
)

Loading CNV-w1a1...

===== CPU BENCHMARK: 5 RUNS =====
Run 1: Latency = 23.2604 ms, Throughput = 42.99 FPS
Run 2: Latency = 25.0271 ms, Throughput = 39.96 FPS
Run 3: Latency = 22.4104 ms, Throughput = 44.62 FPS
Run 4: Latency = 23.9648 ms, Throughput = 41.73 FPS
Run 5: Latency = 22.7999 ms, Throughput = 43.86 FPS

        FINAL CPU RESULTS
Runs                     : 5
Iterations per run       : 500
Average latency          : 23.4925 ms
Latency std between runs : 0.9255 ms
Average throughput       : 42.63 FPS
Throughput std           : 1.65 FPS


In [34]:
print("========== FINAL PRE-CHECK ==========")

print("Folded model:",
      os.path.isfile(folded_model_file))

print("Verification input:",
      os.path.isfile(verify_input_file))

print("Expected output:",
      os.path.isfile(verify_expected_file))

print("Parent:",
      os.path.isfile(verification_parent))

print()
print("Folded model:", folded_model_file)
print("Verification output:", verify_output_dir)

print()
print("Input shape:",
      np.load(verify_input_file).shape)

print("Expected shape:",
      np.load(verify_expected_file).shape)

print("Expected class:",
      np.load(verify_expected_file))

========== FINAL PRE-CHECK ==========
Folded model: True
Verification input: True
Expected output: True
Parent: True

Folded model: /tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_folded.onnx
Verification output: /tmp/finn_dev_kinanah/cnv_folding_C/verification_build

Input shape: (1, 3, 32, 32)
Expected shape: (1,)
Expected class: [3]


In [24]:
model = ModelWrapper(build_dir + "/end2end_cnv_w1a1_dataflow_model.onnx")
fc_layers = model.get_nodes_by_op_type("MVAU_hls")
# each tuple is (PE, SIMD, in_fifo_depth) for a layer
folding = [
    (16, 3, [128]),
    (32, 32, [128]),
    (16, 32, [128]),
    (16, 32, [128]),
    (4, 32, [81]),
    (1, 32, [2]),
    (1, 8, [2]), 
    (1, 16, [128]),
    (5, 1, [3]),
]

from qonnx.custom_op.registry import getCustomOp
from qonnx.core.modelwrapper import ModelWrapper

model_check = ModelWrapper(
    build_dir + "/end2end_cnv_w1a1_dataflow_model.onnx"
)

mvaus = model_check.get_nodes_by_op_type("MVAU_hls")
m6 = getCustomOp(mvaus[7])

MW = m6.get_nodeattr("MW")
MH = m6.get_nodeattr("MH")

print("MVAU_hls_7")
print("MW =", MW)
print("MH =", MH)

valid_simd = [x for x in range(1, MW + 1) if MW % x == 0]
valid_pe   = [x for x in range(1, MH + 1) if MH % x == 0]

print("Valid SIMD values =", valid_simd)
print("Valid PE values =", valid_pe)

for fcl, (pe, simd, ififodepth) in zip(fc_layers, folding):
    fcl_inst = getCustomOp(fcl)
    fcl_inst.set_nodeattr("PE", pe)
    fcl_inst.set_nodeattr("SIMD", simd)
    fcl_inst.set_nodeattr("inFIFODepths", ififodepth)

# use same SIMD values for the sliding window operators
swg_layers = model.get_nodes_by_op_type("ConvolutionInputGenerator_rtl")
for i in range(len(swg_layers)):
    swg_inst = getCustomOp(swg_layers[i])
    simd = folding[i][1]
    swg_inst.set_nodeattr("SIMD", simd)

model = model.transform(GiveUniqueNodeNames())
model.save(build_dir + "/end2end_cnv_w1a1_folded.onnx")

MVAU_hls_7
MW = 512
MH = 512
Valid SIMD values = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
Valid PE values = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]


Below we visualize in Netron to observe the folding factors in the `PE` and `SIMD` attributes of each `MVAU_hls`.

In [26]:
showInNetron(build_dir + "/end2end_cnv_w1a1_folded.onnx")

Serving '/tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_folded.onnx' at http://0.0.0.0:8081


Our network is now ready and we can start with the hardware generation.

In [27]:
import os
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

folded_model_file = os.path.join(build_dir, "end2end_cnv_w1a1_folded.onnx")

perf_output_dir = os.path.join(build_dir, "folding_C_performance")

print("folded_model_file =", folded_model_file)
print("perf_output_dir =", perf_output_dir)

steps = [
    "step_generate_estimate_reports",
    "step_hw_codegen",
    "step_hw_ipgen",
    "step_set_fifo_depths",
    "step_create_stitched_ip",
    "step_measure_rtlsim_performance",
    "step_out_of_context_synthesis",
]

cfg_baseline = build_cfg.DataflowBuildConfig(
    output_dir=perf_output_dir,
    synth_clk_period_ns=target_clk_ns,
    fpga_part=fpga_part,

    # keep the FIFO depths defined in the tutorial baseline
    auto_fifo_depths=False,

    # use several images to measure pipeline throughput
    rtlsim_batch_size=10,
    rtlsim_use_vivado_comps=False,

    steps=steps,

    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
        build_cfg.DataflowOutputType.OOC_SYNTH,
    ],
)

print("Input folded model:", folded_model_file)
print("Performance output:", perf_output_dir)

folded_model_file = /tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_folded.onnx
perf_output_dir = /tmp/finn_dev_kinanah/cnv_folding_C/folding_C_performance
Input folded model: /tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_folded.onnx
Performance output: /tmp/finn_dev_kinanah/cnv_folding_C/folding_C_performance


In [28]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp

m = ModelWrapper(
    build_dir + "/end2end_cnv_w1a1_folded.onnx"
)

results = []

for node in m.get_nodes_by_op_type("MVAU_hls"):
    inst = getCustomOp(node)

    results.append({
        "name": node.name,
        "PE": inst.get_nodeattr("PE"),
        "SIMD": inst.get_nodeattr("SIMD"),
        "cycles": inst.get_exp_cycles()
    })

results = sorted(
    results,
    key=lambda x: x["cycles"],
    reverse=True
)

print("===== Configuration C MVAU cycle estimates =====")

for r in results:
    print(
        r["name"],
        "PE =", r["PE"],
        "SIMD =", r["SIMD"],
        "cycles =", r["cycles"]
    )

print()
print("MVAU bottleneck =", results[0]["name"])
print("Max MVAU cycles =", results[0]["cycles"])

===== Configuration C MVAU cycle estimates =====
MVAU_hls_0 PE = 16 SIMD = 3 cycles = 32400
MVAU_hls_3 PE = 16 SIMD = 32 cycles = 28800
MVAU_hls_1 PE = 32 SIMD = 32 cycles = 28224
MVAU_hls_2 PE = 16 SIMD = 32 cycles = 20736
MVAU_hls_4 PE = 4 SIMD = 32 cycles = 20736
MVAU_hls_5 PE = 1 SIMD = 32 cycles = 18432
MVAU_hls_6 PE = 1 SIMD = 8 cycles = 16384
MVAU_hls_7 PE = 1 SIMD = 16 cycles = 16384
MVAU_hls_8 PE = 5 SIMD = 1 cycles = 1024

MVAU bottleneck = MVAU_hls_0
Max MVAU cycles = 32400


In [36]:
%%time

ret_verify = build.build_dataflow_cfg(
    folded_model_file,
    cfg_verify
)

print("Verification build return code:", ret_verify)

Running step: step_hw_codegen [1/4]


Building dataflow accelerator from /tmp/finn_dev_kinanah/cnv_folding_C/end2end_cnv_w1a1_folded.onnx
Intermediate outputs will be generated in /tmp/finn_dev_kinanah
Final outputs will be generated in /tmp/finn_dev_kinanah/cnv_folding_C/verification_build
Build log is at /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/build_dataflow.log
Running step: step_hw_codegen [1/4]


Running step: step_hw_ipgen [2/4]


Running step: step_hw_ipgen [2/4]


Running step: step_set_fifo_depths [3/4]


Running step: step_set_fifo_depths [3/4]


/home/kinanah/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for Thresholding_rtl_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/kinanah/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for ConvolutionInputGenerator_rtl_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/kinanah/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for MVAU_hls_0
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/kinanah/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for ConvolutionInputGenerator_rtl_1
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/kinanah/finn/src/finn/transformation/fpgadataflow/prepare_ip.py:56: UserWarning: Using pre-existing code for MVAU_hls_1
  warnings.warn("Using pre-existing code for %s" % node.name)
/home/kina

Running step: step_create_stitched_ip [4/4]


creating /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/stitched_ip
creating /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/stitched_ip/finn_vivado_stitch_proj.gen
creating /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/stitched_ip/finn_vivado_stitch_proj.gen/sources_1
creating /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd
creating /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design
copying /tmp/finn_dev_kinanah/vivado_stitch_proj_xjnwkyxl/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/finn_design.bda -> /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design
creating /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ipshared
creating /tmp/finn_dev_kinanah/cnv_folding_C/verification_build/stitched

Completed successfully
Verification build return code: 0
CPU times: user 1min 25s, sys: 4.81 s, total: 1min 30s
Wall time: 28min 57s


In [37]:
import os
import numpy as np

verification_result_dir = os.path.join(
    verify_output_dir,
    "verification_output"
)

print("Verification directory:")
print(verification_result_dir)
print()

if os.path.isdir(verification_result_dir):
    print("Files:")
    for f in os.listdir(verification_result_dir):
        print(" -", f)
else:
    print("Verification directory not found!")

Verification directory:
/tmp/finn_dev_kinanah/cnv_folding_C/verification_build/verification_output

Files:
 - verify_stitched_ip_rtlsim_0_SUCCESS.npy


In [39]:
success_files = [
    f for f in os.listdir(verification_result_dir)
    if "SUCCESS" in f
]

print("SUCCESS files:", success_files)

if len(success_files) > 0:
    rtl_output_file = os.path.join(
        verification_result_dir,
        success_files[0]
    )

    rtl_output = np.load(rtl_output_file)
    expected = np.load(verify_expected_file)

    print("\n===== FUNCTIONAL VERIFICATION RESULT =====")
    print("Golden expected output :", expected)
    print("RTL simulated output   :", rtl_output)

    match = np.isclose(
        expected.reshape(-1),
        rtl_output.reshape(-1),
        atol=1e-3
    ).all()

    print("Match:", match)

    if match:
        print("\nConfiguration C Functional Verification PASSED")
    else:
        print("\n Outputs do not match")

SUCCESS files: ['verify_stitched_ip_rtlsim_0_SUCCESS.npy']

===== FUNCTIONAL VERIFICATION RESULT =====
Golden expected output : [3]
RTL simulated output   : [3.]
Match: True

Configuration C Functional Verification PASSED


## 4. Hardware Generation

From this point onward, the steps we have to follow do not depend on the particular network and will be exactly the same as the TFC-w1a1 example. **which may take about 30 minutes depending on your host computer**. For more details about what's going on in this step, please consult the [TFC end-to-end notebook](tfc_end2end_example.ipynb) or the appropriate section in the [FINN documentation](https://finn.readthedocs.io/en/latest/hw_build.html).

In [23]:
import os, shutil

os.environ["FINN_XILINX_PATH"] = "/tools/Xilinx"
os.environ["FINN_XILINX_VERSION"] = "2022.2"
os.environ["HLS_PATH"] = "/tools/Xilinx/Vitis_HLS/2022.2"
os.environ["VIVADO_PATH"] = "/tools/Xilinx/Vivado/2022.2"
os.environ["PATH"] = (
    os.environ["HLS_PATH"] + "/bin:" +
    os.environ["VIVADO_PATH"] + "/bin:" +
    os.environ["PATH"]
)

print("FINN_XILINX_PATH =", os.getenv("FINN_XILINX_PATH"))
print("FINN_XILINX_VERSION =", os.getenv("FINN_XILINX_VERSION"))
print("HLS_PATH =", os.getenv("HLS_PATH"))
print("VIVADO_PATH =", os.getenv("VIVADO_PATH"))
print("vitis_hls =", shutil.which("vitis_hls"))

FINN_XILINX_PATH = /tools/Xilinx
FINN_XILINX_VERSION = 2022.2
HLS_PATH = /tools/Xilinx/Vitis_HLS/2022.2
VIVADO_PATH = /tools/Xilinx/Vivado/2022.2
vitis_hls = /tools/Xilinx/Vitis_HLS/2022.2/bin/vitis_hls


In [24]:
print(build_dir)
print(pynq_board)
print(target_clk_ns)

/tmp/finn_dev_kinanah/cnv_baseline
Pynq-Z1
10


In [ ]:
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
model = ModelWrapper(build_dir+"/end2end_cnv_w1a1_folded.onnx")
model = model.transform(ZynqBuild(platform = pynq_board, period_ns = target_clk_ns))

After the `ZynqBuild` we run one additional transformation to generate a PYNQ driver for the accelerator.

In [ ]:
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
model = model.transform(MakePYNQDriver("zynq-iodma"))

In [ ]:
model.save(build_dir + "/end2end_cnv_w1a1_synth.onnx")

## 5. Deployment and Execution

The bitfile and generated driver files(s) will be copied into a deployment folder which then can be used to run the network on the PYNQ board.

In [ ]:
from shutil import copy
from distutils.dir_util import copy_tree

# create directory for deployment files
deployment_dir = make_build_dir(prefix="pynq_deployment_")
model.set_metadata_prop("pynq_deployment_dir", deployment_dir)

# get and copy necessary files
# .bit and .hwh file
bitfile = model.get_metadata_prop("bitfile")
hwh_file = model.get_metadata_prop("hw_handoff")
deploy_files = [bitfile, hwh_file]

for dfile in deploy_files:
    if dfile is not None:
        copy(dfile, deployment_dir)

# driver.py and python libraries
pynq_driver_dir = model.get_metadata_prop("pynq_driver_dir")
copy_tree(pynq_driver_dir, deployment_dir)

Next to these files, we will also need an example numpy array to test the network on the PYNQ board. (*and before you ask, that's supposed to be a cat (CIFAR-10 class number 3)*) Recall that we partitioned our original network into a parent graph that contained the non-synthesizable nodes and a child graph that contained the bulk of the network, which we turned into a bitfile. The only operator left outside the FPGA partition was a `Transpose` to convert NCHW images into NHWC ones. Thus, we can skip the execution in the parent as long as we ensure our image has the expected data layout. The example numpy array can then be saved as .npy file.

In [ ]:
import importlib_resources
import matplotlib.pyplot as plt
import numpy as np

ref = importlib_resources.files("finn.qnn-data") / "cifar10/cifar10-test-data-class3.npz"
with importlib_resources.as_file(ref) as fn:
    x = np.load(fn)["arr_0"]
x = x.reshape(3, 32,32).transpose(1, 2, 0)
plt.imshow(x)

In [ ]:
model = ModelWrapper(build_dir + "/end2end_cnv_w1a1_synth.onnx")
iname = model.graph.input[0].name
ishape = model.get_tensor_shape(iname)
np.save(deployment_dir + "/input.npy", x.reshape(ishape))

In [ ]:
! ls {deployment_dir}

In [ ]:
from shutil import make_archive
make_archive('deploy-on-pynq-cnv', 'zip', deployment_dir)

You can now download the created zipfile (File -> Open, mark the checkbox next to the deploy-on-pynq-tfc.zip and select Download from the toolbar), then copy it to your PYNQ board (for instance via scp or rsync). Then, run the following commands on the PYNQ board to extract the archive and run the execution:

```shell
unzip deploy-on-pynq-cnv.zip -d finn-cnv-demo
cd finn-cnv-demo
sudo python3 -m pip install bitstring
sudo python3 driver.py --exec_mode=execute --batchsize=1 --bitfile=resizer.bit --inputfile=input.npy
```

The output will be saved on the PYNQ board as `output.npy` and can be copied to the host and opened with `np.load()`.

### Validating the Accuracy on a PYNQ Board <a id='validation'></a>

All the command line prompts here are meant to be executed with `sudo` on the PYNQ board.

**Ensure that your PYNQ board has a working internet connecting for the next steps, since some there is some downloading involved.**

To validate the accuracy, we first need to install the [`dataset-loading`](https://github.com/fbcotter/dataset_loading) Python package to the PYNQ board. This will give us a convenient way of downloading and accessing the MNIST dataset.


Command to execute on PYNQ:

```shell
sudo pip3 install git+https://github.com/fbcotter/dataset_loading.git@0.0.4#egg=dataset_loading
```

We can now use the `validate.py` script that was generated together with the driver to measure top-1 accuracy on the CIFAR-10 dataset.

Command to execute on PYNQ:

```shell
sudo python3 validate.py --dataset cifar10 --batchsize 1000
```

We see that the final top-1 accuracy is 84.19%, which is very close to the 84.22% reported on the [BNN-PYNQ accuracy table in Brevitas](https://github.com/Xilinx/brevitas/tree/master/src/brevitas_examples/bnn_pynq). 

In [ ]:
import time
import torch
from finn.util.test import get_test_model_trained

torch.set_num_threads(1)  # optional: improves repeatability

print("Loading original Brevitas CNV model...")
cnv_cpu = get_test_model_trained("CNV", 1, 1)
cnv_cpu.eval()

dummy_input = torch.randn(1, 3, 32, 32)

# Warm-up
with torch.no_grad():
    for _ in range(20):
        _ = cnv_cpu(dummy_input)

# Measure
iterations = 100
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(iterations):
        _ = cnv_cpu(dummy_input)
t1 = time.perf_counter()

total_time = t1 - t0
avg_latency_ms = (total_time / iterations) * 1000.0
fps = iterations / total_time

print(f"\n=====================================")
print(f"   תוצאות בסיס השוואה (CPU Baseline)  ")
print(f"=====================================")
print(f"Total time for {iterations} images: {total_time:.4f} seconds")
print(f"Average Latency per image: {avg_latency_ms:.3f} milliseconds")
print(f"Throughput: {fps:.2f} images/second")

In [25]:
import os
import json

report_dir = os.path.join(
    build_dir,
    "folding_B_performance",
    "report"
)

est_path = os.path.join(
    report_dir,
    "estimate_network_performance.json"
)

rtlsim_path = os.path.join(
    report_dir,
    "rtlsim_performance.json"
)

ooc_path = os.path.join(
    report_dir,
    "ooc_synth_and_timing.json"
)

print("Report directory:", report_dir)
print()

# =========================
# Estimated performance
# =========================

if os.path.exists(est_path):
    with open(est_path) as f:
        est = json.load(f)

    print("===== ESTIMATED PERFORMANCE =====")
    for k, v in est.items():
        print(f"{k}: {v}")
else:
    print("estimate_network_performance.json NOT FOUND")

print()

# =========================
# RTL simulation
# =========================

if os.path.exists(rtlsim_path):
    with open(rtlsim_path) as f:
        rtl = json.load(f)

    print("===== RTL SIMULATION =====")

    for key in [
        "latency_cycles",
        "cycles",
        "runtime[ms]",
        "throughput[images/s]",
        "stable_throughput[images/s]",
        "fclk[mhz]",
    ]:
        if key in rtl:
            print(f"{key}: {rtl[key]}")
else:
    print("rtlsim_performance.json NOT FOUND")

print()

# =========================
# OOC synthesis
# =========================

if os.path.exists(ooc_path):
    with open(ooc_path) as f:
        ooc = json.load(f)

    print("===== OOC SYNTHESIS =====")
    for k, v in ooc.items():
        print(f"{k}: {v}")
else:
    print("ooc_synth_and_timing.json NOT FOUND")

Report directory: /tmp/finn_dev_kinanah/cnv_baseline/baseline_performance/report

===== ESTIMATED PERFORMANCE =====
critical_path_cycles: 249362
max_cycles: 32768
max_cycles_node_name: MVAU_hls_6
estimated_throughput_fps: 3051.7578125
estimated_latency_ns: 2493620

===== RTL SIMULATION =====
latency_cycles: 135960
cycles: 579489
runtime[ms]: 5.7948900000000005
throughput[images/s]: 1725.658295498275
stable_throughput[images/s]: 2254.6440029851483
fclk[mhz]: 100.0

===== OOC SYNTHESIS =====
vivado_proj_folder: /tmp/finn_dev_kinanah/synth_out_of_context_x4kt49z6/results_finn_design_wrapper
LUT: 20466.0
LUTRAM: 1034.0
FF: 27932.0
DSP: 0.0
BRAM: 98.0
BRAM_18K: 13.0
BRAM_36K: 92.0
URAM: 0.0
Carry: 2078.0
WNS: 0.461
Delay: 0.461
vivado_version: 2022.2
vivado_build_no: 3671981.0
: 0
fmax_mhz: 104.8327916972429
estimated_throughput_fps: 3199.2429106824616
